# Community Detection with Label Propagation
---

## 1. Introduction

Community detection is an unsupervised learning task that identifies groups of nodes in a graph such that nodes are more strongly connected within groups than across groups.

Unlike clustering in ordinary feature space, community detection works directly with **graph structure**. Instead of distances between feature vectors, the algorithm studies how nodes are linked through edges.

In this notebook, we apply **Label Propagation** using a from-scratch implementation from the `rice_ml` package. The example uses a real network dataset downloaded directly from the web, so no local CSV file is required.

## 2. Mathematical Intuition Behind Community Detection

A graph consists of:

- a set of nodes $V$
- a set of edges $E$ connecting pairs of nodes

The goal is to partition the graph into communities so that:

- connections are dense within a community
- connections are relatively sparse between communities

### Label Propagation Algorithm

Label Propagation is a simple iterative method:

1. Give every node its own unique label
2. For each node, inspect the labels of its neighbors
3. Replace the node's label with the most common neighboring label
4. Repeat until the labels stop changing or a maximum number of iterations is reached

For node $i$, the update rule can be written as

$$
\ell_i^{(t+1)} = \arg\max_{\ell} \sum_{j \in N(i)} \mathbf{1}(\ell_j^{(t)} = \ell)
$$

where $N(i)$ is the neighborhood of node $i$.

Labels spread through highly connected regions, so communities emerge naturally from the graph topology.

## 3. Dataset Overview

This notebook uses the **Facebook Large Page-Page Network** dataset from the UCI Machine Learning Repository. We download the zipped dataset directly from the web and extract the needed files in memory.

Main files used in this example:

### `musae_facebook_edges.csv`

This file defines the graph structure.

- each row is an undirected edge
- `id_1` and `id_2` identify the two connected nodes

### `musae_facebook_target.csv`

This file contains page-level metadata and labels.

- one row per node
- includes page category information such as `page_type`

These labels are not required by Label Propagation itself, but they are useful for interpretation.

In [ ]:
import io
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rice_ml.unsupervised_learning.community_detection import LabelPropagation

In [ ]:
UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/527/facebook%2Blarge%2Bpage%2Bpage%2Bnetwork.zip"

with urllib.request.urlopen(UCI_ZIP_URL) as resp:
    zip_bytes = resp.read()

z = zipfile.ZipFile(io.BytesIO(zip_bytes))

with z.open("facebook_large/musae_facebook_edges.csv") as f:
    edges = pd.read_csv(f)

with z.open("facebook_large/musae_facebook_target.csv") as f:
    targets = pd.read_csv(f)

edges.head(), targets.head()

### Interpretation

At this point we have:

- `edges`: the graph connectivity information
- `targets`: metadata that can help interpret detected communities

## 4. Exploratory Data Analysis (EDA)

The purpose of EDA in community detection is to understand the structure of the graph before running the algorithm.

### Basic Graph Statistics

In [ ]:
n_nodes = int(max(edges["id_1"].max(), edges["id_2"].max()) + 1)
n_edges = len(edges)

n_nodes, n_edges

In [ ]:
deg = np.zeros(n_nodes, dtype=int)

for u, v in edges[["id_1", "id_2"]].itertuples(index=False):
    deg[u] += 1
    deg[v] += 1

deg.min(), deg.max(), deg.mean()

### Interpretation

Social networks often have a heavy-tailed degree distribution. Most nodes have modest degree, while a small number of hubs have many connections. Those hubs often carry useful structural information for community detection.

### Degree Histogram

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(deg, bins=50)
plt.title("Degree Distribution (Full Graph)")
plt.xlabel("Degree")
plt.ylabel("Count")
plt.show()

### Interpretation

Most nodes have relatively small degree, while only a few nodes are highly connected. To make the problem easier to visualize and compute, we next extract a smaller but informative subgraph.

## 5. Building a 500-Node Subgraph

To keep the example computationally manageable, we focus on an induced subgraph formed by the 500 highest-degree nodes.

Steps:

1. Compute degree for each node
2. Select the top 500 nodes by degree
3. Keep only edges whose endpoints are both in that selected set
4. Remap node IDs to consecutive integers
5. Build a dense adjacency matrix

### 1. Compute Node Degrees

In [ ]:
n_nodes = int(max(edges["id_1"].max(), edges["id_2"].max()) + 1)
degrees = np.zeros(n_nodes, dtype=int)

for u, v in edges[["id_1", "id_2"]].itertuples(index=False):
    degrees[u] += 1
    degrees[v] += 1

### 2. Select the Top 500 High-Degree Nodes

In [ ]:
top_k = 500
top_nodes = np.argsort(degrees)[-top_k:]
top_nodes_set = set(top_nodes.tolist())

### 3. Keep Only Edges Within the Selected Node Set

In [ ]:
mask = edges["id_1"].isin(top_nodes_set) & edges["id_2"].isin(top_nodes_set)
edges_sub = edges.loc[mask, ["id_1", "id_2"]].copy()

edges_sub.shape

### 4. Remap Node IDs to a Consecutive Index Set

In [ ]:
old_to_new = {old_id: new_id for new_id, old_id in enumerate(sorted(top_nodes_set))}
n_sub = len(old_to_new)

u_new = edges_sub["id_1"].map(old_to_new).to_numpy()
v_new = edges_sub["id_2"].map(old_to_new).to_numpy()

### 5. Construct the Dense Adjacency Matrix

In [ ]:
A = np.zeros((n_sub, n_sub), dtype=int)

for u, v in zip(u_new, v_new):
    A[u, v] = 1
    A[v, u] = 1

np.fill_diagonal(A, 0)

A.shape

## 6. Community Detection via Label Propagation

We now apply Label Propagation to the induced subgraph represented by adjacency matrix `A`.

Label Propagation does not need a predefined number of communities. Instead, the number and size of communities are determined by the connectivity pattern of the graph itself.

In [ ]:
lp = LabelPropagation(max_iter=100)
labels = lp.fit_predict(A)

## 7. Understanding Community Labels

Community labels are identifiers only.

- the numeric values are not ordered
- only equality matters
- the number of distinct labels gives the number of detected communities

In [ ]:
unique_labels, counts = np.unique(labels, return_counts=True)
community_sizes = dict(zip(unique_labels, counts))
community_sizes

### Interpretation

A few large communities may indicate dominant structural groups, while several small communities may reflect more localized patterns. Because the algorithm is unsupervised, this structure is learned directly from the graph.

## 8. Connectivity vs Community Assignment

In [ ]:
degrees_sub = A.sum(axis=1)

plt.figure(figsize=(6, 4))
plt.scatter(degrees_sub, labels, s=10)
plt.xlabel("Node Degree (Subgraph)")
plt.ylabel("Community Label")
plt.title("Degree vs Community Assignment")
plt.show()

### Interpretation

Highly connected nodes often act as anchors for communities. Lower-degree nodes may attach to nearby groups or remain in smaller communities.

## 9. Community Detection Visualization

To visualize the detected communities, we use a spectral embedding based on the graph Laplacian.

### Graph Laplacian

If $A$ is the adjacency matrix and $D$ is the degree matrix, then the unnormalized Laplacian is

$$
L = D - A
$$

The eigenvectors associated with the smallest nonzero eigenvalues provide a low-dimensional representation that preserves graph structure.

### Compute the Laplacian

In [ ]:
degrees_sub = A.sum(axis=1)
D = np.diag(degrees_sub)
L = D - A

### Spectral Embedding (2D)

In [ ]:
evals, evecs = np.linalg.eigh(L)
idx = np.argsort(evals)

coords = evecs[:, idx[1:3]]

## 10. Visualizing Communities

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(coords[:, 0], coords[:, 1], c=labels, s=18)
plt.xlabel("Spectral Coordinate 1")
plt.ylabel("Spectral Coordinate 2")
plt.title("Community Detection Visualization")
plt.show()

### Interpretation

Nodes that appear close together in the spectral embedding tend to have similar graph connectivity. Coloring by community label helps reveal whether the Label Propagation result aligns with the graph structure visible in the embedding.

## 11. Comparing Detected Communities to Known Page Types

Although Label Propagation does not use page labels, we can compare the detected communities to the known `page_type` metadata for interpretation.

In [ ]:
selected_old_ids = np.array(sorted(top_nodes_set))
selected_targets = targets[targets["id"].isin(selected_old_ids)].copy()

selected_targets["sub_id"] = selected_targets["id"].map(old_to_new)
selected_targets = selected_targets.sort_values("sub_id")

selected_targets["community_label"] = labels
selected_targets[["id", "page_type", "community_label"]].head()

In [ ]:
community_page_counts = (
    selected_targets.groupby(["community_label", "page_type"])
    .size()
    .unstack(fill_value=0)
)

community_page_counts.head()

### Interpretation

This summary helps us see whether communities correspond to recognizable content types. Strong concentration of one page type inside a community suggests that the graph structure is capturing meaningful real-world grouping.

## 12. Conclusion

In this notebook, we:

- downloaded a real graph dataset directly from the web
- built a high-degree induced subgraph
- applied from-scratch Label Propagation
- examined community sizes and degree relationships
- visualized the detected communities with spectral embedding
- compared communities to known page metadata

This example shows that community detection can recover meaningful graph structure without using traditional feature-space clustering.